# مترجم خودکار مانگا / مانهوا به فارسی

OCR → پاک‌سازی حباب → ترجمه با Gemini → رندر فارسی

**ترتیب:** سلول‌ها را از بالا به پایین با Shift+Enter اجرا کنید.

قبل از شروع: `Runtime → Change runtime type → GPU (T4)`

## ۴) نوشتن اسکریپت اصلی مترجم
این سلول فایل `manga_translator.py` رو می‌سازه (همون پایپ‌لاین: تشخیص متن -> پاکسازی -> ترجمه -> بازنویسی).

In [1]:
!git clone https://github.com/amirwolf5122/Manga-AutoTranslate.git
!cp Manga-AutoTranslate/manga_translator.py .
!rm -rf Manga-AutoTranslate

Telegram:
@Amir_wolf512


## ۱) نصب پیش‌نیازها

In [1]:
# ساخت فایل constraints
with open("constraints.txt", "w") as f:
    f.write("""numpy==1.26.4
opencv-python-headless==4.8.1.78
opencv-python==4.8.1.78
opencv-contrib-python==4.8.1.78
""")

# نصب پکیج‌ها
!pip install --no-cache-dir --constraint constraints.txt numpy==1.26.4
!pip install --no-cache-dir --constraint constraints.txt opencv-python-headless==4.8.1.78
!pip install --no-cache-dir --no-deps --constraint constraints.txt paddlepaddle==2.6.2 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
!pip install --no-cache-dir --no-deps --constraint constraints.txt paddleocr==2.7.0.3
!pip install --no-cache-dir --constraint constraints.txt pymupdf
!pip install --no-cache-dir --constraint constraints.txt attrdict cython fire lxml openpyxl pdf2docx premailer python-docx visualdl
!pip install --no-cache-dir --constraint constraints.txt Pillow pyclipper lmdb scikit-image shapely python-bidi arabic-reshaper rapidfuzz imageio matplotlib tqdm requests beautifulsoup4 google-genai decorator imgaug opt-einsum astor pyyaml simple-lama-inpainting

## ۲) دانلود فونت فارسی (Vazirmatn)
برای اینکه متن فارسی درست نمایش داده بشه، یک فونت پشتیبان فارسی لازم داریم.

In [2]:
import os
os.makedirs('fonts', exist_ok=True)
!wget -q -O fonts/Vazirmatn-Bold.ttf \
  https://github.com/rastikerdar/vazirmatn/raw/master/fonts/ttf/Vazirmatn-Bold.ttf
print('فونت دانلود شد:', os.path.isfile('fonts/Vazirmatn-Bold.ttf'))

## ۳) کلید Gemini API
کلید رایگان‌تون رو از [aistudio.google.com/api-keys](https://aistudio.google.com/api-keys) بگیرید و اینجا (به‌صورت مخفی) وارد کنید.

In [7]:
from getpass import getpass
import os

print("کلیدهای Gemini رو یکی‌یکی وارد کن (خالی بذار تا تموم بشه):")
keys = []
while True:
    k = getpass(f"کلید {len(keys)+1} (Enter = پایان): ").strip()
    if not k:
        break
    keys.append(k)

if not keys:
    raise SystemExit("حداقل یک کلید لازم است.")

os.environ["GEMINI_API_KEY"] = ",".join(keys)
print(f"{len(keys)} کلید ثبت شد.")

زبان اصلی متن منبع رو انتخاب کنید (این مهمه؛ انتخاب اشتباه باعث می‌شه OCR متن رو درست استخراج نکنه):

In [5]:
print("زبان اصلی متن منبع رو انتخاب کنید:")
print(" 1) en (انگلیسی - اکثر اسکنلیشن‌ها)")
print(" 2) ja en (ژاپنی خام)")
print(" 3) ko en (کره‌ای خام)")
print(" 4) دستی وارد کنید")

lang_choice = input("انتخاب [پیش‌فرض 1]: ").strip() or "1"

if lang_choice == "1":
    OCR_LANG = "en"
elif lang_choice == "2":
    OCR_LANG = "ja en"
elif lang_choice == "3":
    OCR_LANG = "ko en"
elif lang_choice == "4":
    OCR_LANG = input("زبان OCR را وارد کنید: ").strip()
else:
    OCR_LANG = "en"

print(f"زبان OCR: {OCR_LANG}")
print()

## ۵) ورودی رو بدید و ترجمه رو اجرا کنید
این سلول خودش تشخیص می‌ده که چی بهش دادید:
- اگه یک **لینک** (http/https) وارد کنید، تصاویر همون صفحه خودکار دانلود می‌شن.
- اگه Enter بزنید، پنجره‌ی آپلود باز می‌شه؛ می‌تونید یک فایل **.zip**، یک فایل **.pdf**، یا چند تا **تصویر** (jpg/png/...) رو هم‌زمان انتخاب کنید — نوعش خودکار تشخیص داده می‌شه.

In [4]:
import os
from google.colab import files

INPUT_DIR = 'input_pages'
os.makedirs(INPUT_DIR, exist_ok=True)

url = input('اگه لینک صفحه دارید وارد کنید (وگرنه Enter بزنید تا فایل آپلود کنید): ').strip()

if url.lower().startswith('http://') or url.lower().startswith('https://'):
    input_path = url
    print(f'از لینک استفاده می‌شه: {input_path}')
else:
    print('فایل(ها) رو انتخاب کنید (یک zip، یک pdf، یا چند تصویر):')
    uploaded = files.upload()
    names = list(uploaded.keys())

    if len(names) == 1 and names[0].lower().endswith('.zip'):
        input_path = names[0]
        with open(input_path, 'wb') as f:
            f.write(uploaded[names[0]])
        print(f'فایل zip شناسایی و ذخیره شد: {input_path}')

    elif len(names) == 1 and names[0].lower().endswith('.pdf'):
        input_path = names[0]
        with open(input_path, 'wb') as f:
            f.write(uploaded[names[0]])
        print(f'فایل pdf شناسایی و ذخیره شد: {input_path}')

    else:
        for name, data in uploaded.items():
            with open(os.path.join(INPUT_DIR, name), 'wb') as f:
                f.write(data)
        input_path = INPUT_DIR
        print(f'{len(names)} تصویر آپلود و در پوشه‌ی {INPUT_DIR} ذخیره شد.')

print('\nورودی نهایی برای پردازش:', input_path)

خروجی پیش‌فرض **PDF** است و نام فایل خودکار از روی لینک یا فایل ورودی ساخته می‌شود (مثلاً `eu39-green-skin-chapter-13-eng-li.pdf`).

برای مانگای ژاپنی خام `--ocr-lang ja en`، برای کره‌ای `ko en`، برای انگلیسی اسکنلیشن فقط `en`.
برای کمیک چپ‌به‌راست، `--reading-order ltr` بگذارید.


In [8]:
import os
import re
from urllib.parse import urlparse, unquote

out_ext = '.pdf'

def _auto_output_path(input_path: str, output_spec: str) -> str:
    spec = (output_spec or '').strip()
    is_ext_only = (
        spec.startswith('.')
        and '/' not in spec and '\\' not in spec
        and re.fullmatch(r'\.(pdf|zip|html)', spec, re.I) is not None
    )
    if not is_ext_only:
        return output_spec
    ext = spec.lower()
    if input_path.lower().startswith(('http://', 'https://')):
        path = unquote(urlparse(input_path).path).strip('/')
        parts = [p for p in path.split('/') if p]
        base = 'chapter'
        if parts and 'chapter' in [p.lower() for p in parts]:
            low = [p.lower() for p in parts]
            try:
                idx = low.index('chapter')
                name = parts[idx - 1] if idx > 0 else 'chapter'
                num = parts[idx + 1] if idx + 1 < len(parts) else ''
                num = re.sub(r'[^\w\-]', '', num.split('?')[0])
                base = f'{name}-{num}' if num else name
            except ValueError:
                base = parts[-1]
        elif parts:
            base = parts[-1]
        base = re.sub(r'[^\w\-.]+', '-', base).strip('-._') or 'chapter'
    else:
        raw = input_path.rstrip('/\\')
        base = os.path.splitext(os.path.basename(raw))[0] or 'output'
        base = re.sub(r'[^\w\-.]+', '-', base).strip('-._') or 'output'
    return base + ext

output_path = _auto_output_path(input_path, out_ext)
print(f'خروجی نهایی: {output_path}')

!python manga_translator.py \
  -i "{input_path}" \
  -o "{out_ext}" \
  --font fonts/Vazirmatn-Bold.ttf \
  --ocr-lang "{OCR_LANG}" \
  --reading-order rtl

if not os.path.isfile(output_path):
    candidates = [f for f in os.listdir('.') if f.lower().endswith(out_ext)]
    if candidates:
        candidates.sort(key=lambda f: os.path.getmtime(f), reverse=True)
        output_path = candidates[0]
        print(f'فایل خروجی پیدا شد: {output_path}')
    else:
        print('هشدار: فایل خروجی پیدا نشد.')
else:
    print(f'فایل خروجی آماده است: {output_path}')

# نکته: --ocr-lang باید با زبانی که روی خود تصویر چاپ شده یکی باشه.
# اگه فایلتون از قبل انگلیسی اسکنلیشن شده (اکثر ریلیزهای معروف اینطورن)،
# 'en' درسته. برای اسکن خام ژاپنی از 'ja en' و برای کره‌ای از 'ko en' استفاده کنید.

# اگه این سلول قطع شد (مثلاً سهمیه‌ی روزانه‌ی Gemini تموم شد)، فقط دوباره
# همین سلول رو اجرا کنید؛ صفحاتی که قبلاً ترجمه شدن دوباره پردازش نمی‌شن.


## ۷) دانلود خروجی‌ها

In [ ]:
from google.colab import files
import os
import zipfile
from IPython.display import display, IFrame
from PIL import Image as PILImage

if 'output_path' not in globals() or not output_path:
    for ext in ('.html', '.pdf', '.zip'):
        cands = [f for f in os.listdir('.') if f.lower().endswith(ext)]
        if cands:
            cands.sort(key=lambda f: os.path.getmtime(f), reverse=True)
            output_path = cands[0]
            break
    else:
        raise SystemExit('هیچ فایل خروجی پیدا نشد. اول سلول ترجمه را اجرا کنید.')

print(f'فایل خروجی: {output_path}')
print('انتخاب کنید:')
print('1) دانلود ترجمه')
print('2) نمایش ترجمه')
choice = input('عدد رو وارد کنید [پیش‌فرض 1]: ').strip() or '1'

if str(choice) == '1':
    if not os.path.exists(output_path):
        print(f'فایل یافت نشد: {output_path}')
    else:
        files.download(output_path)
else:
    if not os.path.exists(output_path):
        print(f'فایل یافت نشد: {output_path}')

    elif output_path.lower().endswith('.html'):
        print('در حال نمایش فایل html...')
        display(IFrame(output_path, width='100%', height=900))

    elif output_path.lower().endswith('.pdf'):
        print('در حال نمایش فایل PDF...')
        display(IFrame(output_path, width=800, height=600))

    elif output_path.lower().endswith('.zip'):
        print('در حال استخراج و نمایش عکس‌های درون ZIP...')
        extract_folder = 'temp_extracted_imgs'
        os.makedirs(extract_folder, exist_ok=True)

        with zipfile.ZipFile(output_path, 'r') as zip_ref:
            zip_ref.extractall(extract_folder)

        valid_extensions = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
        image_files = []
        for root, _, files_list in os.walk(extract_folder):
            for f in files_list:
                if f.lower().endswith(valid_extensions):
                    image_files.append(os.path.join(root, f))

        image_files.sort()

        if image_files:
            print(f'تعداد {len(image_files)} تصویر پیدا شد:\n')
            for img_path in image_files:
                print(f' {os.path.basename(img_path)}')
                img = PILImage.open(img_path)
                display(img)
                print('-' * 40)
        else:
            print('هیچ تصویری داخل فایل زیپ پیدا نشد.')

    else:
        print('فرمت فایل پشتیبانی نمی‌شود (فقط .html / .pdf / .zip).')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>